# Notebook A · Authors’ PyramidKV on LongBench TREC

Run stock FullKV and the PyramidKV authors’ Llama SDPA implementation from a [pinned KVCache-Factory commit](https://github.com/Zefan-Cai/KVCache-Factory/commit/68cd9551a63fedf362ddb0edb008badd04c24996) on all 200 LongBench v1 TREC examples. This uses the paper’s Llama 3.2-3B-Instruct AWQ checkpoint, nominal KV budget of 1,024, batch size one, greedy decoding, and a 4,096-token prompt cap. The run saves prompt token IDs, references, predictions, and observed cache lengths for Notebook B.

Open this notebook in a fresh Colab runtime. In **Runtime → Change runtime type**, select **T4 GPU** and **Runtime version 2026.07** (Python 3.12). Then run cells 1 through 5. [Colab runtime-version guide](https://research.google.com/colaboratory/runtime-version-faq.html)

If the legacy AWQ backend fails in setup, loading or the pilot, save the traceback. To run the matched FP16 alternative, use a fresh T4 runtime, set `PRECISION = "fp16"` in step 1, and rerun A. Notebook B reads the selected precision from A’s ZIP. Both arms must use the same checkpoint and precision.

This measures agreement on the 3B experiment’s model. A literal reproduction of the published Llama-3-8B scores would require the published model and settings.

## 1. Select precision and install the upstream environment

This installs the upstream Transformers version for legacy Llama attention and leaves the runtime’s GPU PyTorch installation in place.

In [1]:
import subprocess
import sys

# Edit only this precision setting if you need to run both arms in FP16.
PRECISION = "awq"  # "awq" or "fp16"
NOMINAL_BUDGET = 1024
PROMPT_TOKEN_CAP = 4096
assert PRECISION in ("awq", "fp16")

assert sys.version_info[:2] == (3, 12), (
    "In Colab choose Runtime > Change runtime type > Runtime version 2026.07 (Python 3.12)."
)
gpu_names = subprocess.check_output(
    ["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"], text=True
).splitlines()
assert len(gpu_names) == 1 and "T4" in gpu_names[0], f"Expected one NVIDIA T4, found {gpu_names}."
pip = [sys.executable, "-m", "pip", "install", "-q", "--disable-pip-version-check"]

legacy_packages = [
    "transformers==4.44.2", "tokenizers==0.19.1",
    "huggingface-hub==0.26.5", "accelerate==1.13.0",
]
if PRECISION == "awq":
    legacy_packages += ["datasets==3.6.0", "zstandard"]
subprocess.run([*pip, *legacy_packages], check=True)
if PRECISION == "awq":
    # Avoid installing AutoAWQ's CPU Intel extension in the CUDA runtime.
    subprocess.run([*pip, "--no-deps", "autoawq==0.2.7"], check=True)
print(f"Notebook A dependencies installed for {PRECISION} on {gpu_names[0]}.")

Notebook A dependencies installed for awq on Tesla T4.


## 2. Freeze the 200 TREC inputs

Use the upstream plain-text TREC prompt template. Keep the first and last halves of prompts that exceed the cap. The ZIP records the exact inputs, so B uses the same tokens, case order and class labels.

In [2]:
import ast
import hashlib
import importlib.metadata
import json
import platform
import subprocess
import time
import uuid
from datetime import datetime, timezone
from pathlib import Path
from shutil import copyfile
from tempfile import TemporaryDirectory
from urllib.request import urlopen
from zipfile import ZIP_DEFLATED, ZipFile

import numpy as np
import torch
import transformers
from huggingface_hub import snapshot_download
from transformers import AutoModelForCausalLM, DynamicCache, PreTrainedTokenizerFast

SCHEMA = "paper744-llama-3b-trec-v1"
UPSTREAM_COMMIT = "68cd9551a63fedf362ddb0edb008badd04c24996"
UPSTREAM_URL = "https://github.com/Zefan-Cai/KVCache-Factory.git"
UPSTREAM_RAW = f"https://raw.githubusercontent.com/Zefan-Cai/KVCache-Factory/{UPSTREAM_COMMIT}/"
DATA_PATH = "data/LongBench/trec.jsonl"
DATA_SHA256 = "20a88e14f7442f58a39082cdd7396f1093ea8162da0a4ea7ead7d6757514c8f4"
METRICS_SHA256 = "76cf741219c29ae5e8fd0c1b1c00397f53aa14a32177f0867f1a4ed03aa77f24"
MODEL_SPECS = {
    "awq": ("casperhansen/llama-3.2-3b-instruct-awq", "272b3bde867b606760447deb9a4d2719fbdfd3ae"),
    "fp16": ("unsloth/Llama-3.2-3B-Instruct", "006f5dcd1393c3add266de40994ba96225e9689d"),
}
NUM_LAYERS = 28
NUM_Q_HEADS = 24
NUM_KV_HEADS = 8
HEAD_DIM = 128
CASE_COUNT = 200
MAX_NEW_TOKENS = 64
SEED = 407
ATTENTION_BACKEND = "sdpa"


def digest(data):
    return hashlib.sha256(data).hexdigest()


def object_digest(value):
    encoded = json.dumps(value, sort_keys=True, separators=(",", ":"), allow_nan=False).encode()
    return digest(encoded)


def id_digest(ids):
    return digest(",".join(map(str, ids)).encode("ascii"))


def download_upstream(name, expected_sha256):
    with urlopen(UPSTREAM_RAW + name, timeout=60) as response:
        data = response.read()
    assert digest(data) == expected_sha256, f"Unexpected source hash for {name}"
    return data


def write_json(path, value):
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(json.dumps(value, ensure_ascii=False, allow_nan=False) + "\n", encoding="utf-8")
    temporary.replace(path)


def read_json(path):
    return json.loads(path.read_text(encoding="utf-8"))


def read_jsonl(data):
    return [json.loads(line) for line in data.splitlines() if line.strip()]


def write_jsonl(path, rows):
    temporary = path.with_suffix(path.suffix + ".tmp")
    with temporary.open("w", encoding="utf-8") as out:
        for row in rows:
            out.write(json.dumps(row, ensure_ascii=False, allow_nan=False) + "\n")
    temporary.replace(path)


def start_output_directory(arm):
    stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    run_id = f"{stamp}-{uuid.uuid4().hex[:8]}"
    output = Path("/content") / f"paper744_trec_{arm}_{run_id}"
    output.mkdir()
    return output, run_id


def package_versions():
    values = {"python": platform.python_version(), "torch": str(torch.__version__), "cuda": torch.version.cuda}
    for name in ("transformers", "tokenizers", "accelerate", "autoawq", "gptqmodel", "triton"):
        try:
            values[name] = importlib.metadata.version(name)
        except importlib.metadata.PackageNotFoundError:
            values[name] = None
    return values


def observed_gpu():
    assert torch.cuda.is_available() and torch.cuda.device_count() == 1
    device = torch.cuda.get_device_properties(0)
    assert "T4" in device.name, f"Select a Colab T4 runtime, found {device.name}"
    return {"name": device.name, "memory_mib": device.total_memory // 2**20}


def tokenizer_and_stop_ids(spec):
    names = ("config.json", "tokenizer.json", "tokenizer_config.json", "special_tokens_map.json")
    snapshot = Path(snapshot_download(
        spec["model_id"], revision=spec["model_revision"], allow_patterns=list(names),
    ))
    if transformers.__version__ == "4.44.2":
        # Tokenizers 0.19 reads BPE merges as strings. Convert their serialization
        # without changing the ordered merge pairs, vocabulary, or tokenizer settings.
        with TemporaryDirectory() as directory:
            local = Path(directory)
            for name in ("tokenizer_config.json", "special_tokens_map.json"):
                if (snapshot / name).is_file():
                    copyfile(snapshot / name, local / name)
            tokenizer_data = read_json(snapshot / "tokenizer.json")
            bpe = tokenizer_data["model"]
            assert bpe["type"] == "BPE"
            bpe["merges"] = [" ".join(pair) if isinstance(pair, list) else pair for pair in bpe["merges"]]
            write_json(local / "tokenizer.json", tokenizer_data)
            tokenizer = PreTrainedTokenizerFast.from_pretrained(str(local))
    else:
        tokenizer = PreTrainedTokenizerFast.from_pretrained(str(snapshot))
    model_eos = read_json(snapshot / "config.json")["eos_token_id"]
    if isinstance(model_eos, int):
        model_eos = [model_eos]
    eot = tokenizer.convert_tokens_to_ids("<|eot_id|>")
    assert eot is not None and eot != tokenizer.unk_token_id
    stop_tokens = list(dict.fromkeys([*model_eos, tokenizer.eos_token_id, eot]))
    assert all(type(token) is int and 0 <= token < len(tokenizer) for token in stop_tokens)
    return tokenizer, stop_tokens


def validate_workload(spec, cases):
    assert spec["schema"] == SCHEMA and spec["dataset"] == "LongBench v1 TREC"
    assert spec["dataset_sha256"] == DATA_SHA256
    assert spec["upstream_commit"] == UPSTREAM_COMMIT and spec["metrics_sha256"] == METRICS_SHA256
    assert spec["attention_backend"] == ATTENTION_BACKEND and spec["max_new_tokens"] == MAX_NEW_TOKENS
    assert tuple((spec["model_id"], spec["model_revision"])) == MODEL_SPECS[spec["precision"]]
    assert len(cases) == spec["case_count"] == CASE_COUNT
    assert spec["batch_size"] == 1 and spec["decoding"] == "greedy"
    assert spec["stop_token_ids"] and len(set(spec["stop_token_ids"])) == len(spec["stop_token_ids"])
    assert all(type(token) is int and 0 <= token < spec["vocab_size"]
               for token in [*spec["stop_token_ids"], spec["pad_token_id"]])
    assert [case["index"] for case in cases] == list(range(CASE_COUNT))
    assert len({case["case_id"] for case in cases}) == CASE_COUNT
    assert 32 < spec["budget"] < spec["prompt_token_cap"] <= 7500
    assert all(case["answers"] and set(case["answers"]) <= set(case["all_classes"]) for case in cases)
    for case in cases:
        ids = case["input_ids"]
        assert 0 < len(ids) <= spec["prompt_token_cap"]
        assert all(type(token) is int and 0 <= token < spec["vocab_size"] for token in ids)
        assert id_digest(ids) == case["input_ids_sha256"]


def load_model(spec, expected_transformers):
    assert transformers.__version__ == expected_transformers, "Restart the runtime and rerun setup"
    torch.manual_seed(SEED)
    model = AutoModelForCausalLM.from_pretrained(
        spec["model_id"], revision=spec["model_revision"],
        torch_dtype=torch.float16, device_map={"": 0}, low_cpu_mem_usage=True,
        attn_implementation=ATTENTION_BACKEND, use_safetensors=True,
    ).eval()
    config = model.config
    assert (config.model_type, config.num_hidden_layers, config.num_attention_heads,
            config.num_key_value_heads) == ("llama", NUM_LAYERS, NUM_Q_HEADS, NUM_KV_HEADS)
    assert getattr(config, "head_dim", config.hidden_size // NUM_Q_HEADS) == HEAD_DIM
    assert config.vocab_size == spec["vocab_size"]
    assert config._attn_implementation == ATTENTION_BACKEND
    if spec["precision"] == "awq":
        assert config.quantization_config.to_dict()["quant_method"] == "awq"
        assert config.quantization_config.to_dict()["bits"] == 4
        assert hasattr(model.model.layers[0].self_attn.q_proj, "qweight")
    else:
        assert model.model.layers[0].self_attn.q_proj.weight.dtype == torch.float16
    assert model.get_input_embeddings().weight.device.type == "cuda", "Expected the 3B weights on the T4"
    return model


def model_runtime(model):
    projection = model.model.layers[0].self_attn.q_proj
    cls = type(projection)
    return {"q_proj_class": f"{cls.__module__}.{cls.__name__}", "model_dtype": str(model.dtype)}


def cache_shapes(cache):
    if hasattr(cache, "layers"):  # Transformers 5.x DynamicCache
        layers = [layer.keys for layer in cache.layers]
    elif hasattr(cache, "key_cache"):  # Transformers 4.44 DynamicCache
        layers = cache.key_cache
    else:
        layers = [pair[0] for pair in cache]
    return [(int(key.shape[1]), int(key.shape[-2])) for key in layers]


def build_row(case, mode, generated_ids, kept, final, tokenizer, spec):
    assert generated_ids and len(generated_ids) <= spec["max_new_tokens"]
    assert all(0 <= token < spec["vocab_size"] for token in generated_ids)
    assert len(kept) == len(final) == NUM_LAYERS
    prompt_len = len(case["input_ids"])
    assert all(0 < length <= prompt_len for _, length in kept)
    assert all(heads in (NUM_KV_HEADS, NUM_Q_HEADS) for heads, _ in kept)
    assert all(
        head == final_head and final_len == kept_len + len(generated_ids) - 1
        for (head, kept_len), (final_head, final_len) in zip(kept, final)
    ), "The observed prefill cache was not used through decode"
    if mode == "fullkv":
        assert all(shape == (NUM_KV_HEADS, prompt_len) for shape in kept)
    elif prompt_len > spec["budget"]:
        assert any(length < prompt_len for _, length in kept), "Eviction did not reduce the stored cache"
    return {
        "case_id": case["case_id"], "index": case["index"], "mode": mode,
        "input_ids_sha256": case["input_ids_sha256"],
        "generated_ids": generated_ids,
        "prediction": tokenizer.decode(generated_ids, skip_special_tokens=True, clean_up_tokenization_spaces=False),
        "prefill_cache_tokens": [length for _, length in kept],
        "cache_heads": [heads for heads, _ in kept],
        "final_cache_tokens": [length for _, length in final],
    }


def pilot_cases(cases):
    by_length = sorted(cases, key=lambda case: (len(case["input_ids"]), case["original_token_count"], case["index"]))
    return [by_length[i] for i in (0, len(cases)//4, len(cases)//2, (3*len(cases))//4, len(cases)-1)]


def validate_rows(rows, cases, mode, spec):
    assert len(rows) == len(cases)
    assert [r["case_id"] for r in rows] == [c["case_id"] for c in cases]
    for row, case in zip(rows, cases):
        assert row["mode"] == mode and row["index"] == case["index"]
        assert row["input_ids_sha256"] == case["input_ids_sha256"]
        prompt_len = len(case["input_ids"])
        generated = row["generated_ids"]
        assert len(generated) <= spec["max_new_tokens"] and len(generated) > 0
        assert all(type(t) is int and 0 <= t < spec["vocab_size"] for t in generated)
        assert isinstance(row["prediction"], str)
        prefill = row["prefill_cache_tokens"]
        assert len(prefill) == len(row["final_cache_tokens"]) == len(row["cache_heads"]) == NUM_LAYERS
        assert all(0 < length <= prompt_len for length in prefill)
        assert row["final_cache_tokens"] == [length + len(generated) - 1 for length in prefill]
        assert all(heads in (NUM_Q_HEADS, NUM_KV_HEADS) for heads in row["cache_heads"])
        if mode == "fullkv":
            assert prefill == [prompt_len] * NUM_LAYERS
            assert row["cache_heads"] == [NUM_KV_HEADS] * NUM_LAYERS
        elif prompt_len > spec["budget"]:
            assert min(prefill) < prompt_len


def score_function():
    # Extract the unchanged, dependency-free TREC metric from the authors' pinned metrics.py.
    # See the pinned upstream repository and its MIT license.
    source = download_upstream("metrics.py", METRICS_SHA256)
    tree = ast.parse(source, filename="upstream/metrics.py")
    function = next(node for node in tree.body if isinstance(node, ast.FunctionDef)
                    and node.name == "classification_score")
    namespace = {}
    module = ast.Module(body=[function], type_ignores=[])
    exec(compile(module, "upstream/metrics.py", "exec"), namespace)
    return namespace["classification_score"]


def score_prediction(classifier, prediction, case):
    # The official eval.py scores the first line for TREC and takes the best reference score.
    first_line = prediction.lstrip("\n").split("\n", 1)[0]
    return max(classifier(first_line, answer, all_classes=case["all_classes"])
               for answer in case["answers"])


def make_zip(output, names):
    hashes = {name: digest((output / name).read_bytes()) for name in names}
    write_json(output / "integrity.json", {"file_sha256": hashes})
    path = output.with_suffix(".zip")
    temporary = path.with_suffix(".zip.tmp")
    with ZipFile(temporary, "w", compression=ZIP_DEFLATED) as archive:
        for name in [*names, "integrity.json"]:
            archive.write(output / name, name)
    temporary.replace(path)
    with ZipFile(path) as archive:
        assert archive.testzip() is None and set(archive.namelist()) == set(names) | {"integrity.json"}
    return path


def read_verified_zip(path, names):
    with ZipFile(path) as archive:
        expected = set(names) | {"integrity.json"}
        assert set(archive.namelist()) == expected and len(archive.namelist()) == len(expected)
        assert sum(member.file_size for member in archive.infolist()) < 100_000_000
        integrity = json.loads(archive.read("integrity.json"))["file_sha256"]
        assert set(integrity) == set(names)
        result = {name: archive.read(name) for name in names}
    assert all(digest(data) == integrity[name] for name, data in result.items())
    return result


OFFICIAL_FILES = ("workload.json", "cases.jsonl", "run_manifest.json", "fullkv.jsonl", "official_pyramidkv.jsonl")

In [3]:
GPU = observed_gpu()
assert transformers.__version__ == "4.44.2"
assert 32 < NOMINAL_BUDGET < PROMPT_TOKEN_CAP <= 7500
OUTPUT, RUN_ID = start_output_directory("official")
model_id, model_revision = MODEL_SPECS[PRECISION]
SPEC = {
    "schema": SCHEMA, "dataset": "LongBench v1 TREC", "dataset_sha256": DATA_SHA256,
    "upstream_commit": UPSTREAM_COMMIT, "metrics_sha256": METRICS_SHA256,
    "precision": PRECISION, "model_id": model_id, "model_revision": model_revision,
    "attention_backend": ATTENTION_BACKEND, "budget": NOMINAL_BUDGET,
    "prompt_token_cap": PROMPT_TOKEN_CAP, "max_new_tokens": MAX_NEW_TOKENS,
    "case_count": CASE_COUNT, "batch_size": 1, "decoding": "greedy",
    "truncation": "keep first and last token spans of the tokenized TREC prompt",
}
TOKENIZER, SPEC["stop_token_ids"] = tokenizer_and_stop_ids(SPEC)
SPEC["vocab_size"] = len(TOKENIZER)
SPEC["pad_token_id"] = (TOKENIZER.pad_token_id if TOKENIZER.pad_token_id is not None
                        else TOKENIZER.eos_token_id)
raw_rows = read_jsonl(download_upstream(DATA_PATH, DATA_SHA256))
CASES = []
for index, example in enumerate(raw_rows):
    # Same plain TREC template as the pinned runner. Truncation is applied to token IDs
    # once here, then the exact tensors are shared across both software environments.
    prompt = ("Please determine the type of the question below. Here are some examples of questions."
              f'\n\n{example["context"]}\n{example["input"]}')
    ids = TOKENIZER.encode(prompt, add_special_tokens=True, truncation=False)
    original_length = len(ids)
    if original_length > PROMPT_TOKEN_CAP:
        left = PROMPT_TOKEN_CAP // 2
        ids = ids[:left] + ids[-(PROMPT_TOKEN_CAP - left):]
    CASES.append({
        "index": index, "case_id": example["_id"], "original_token_count": original_length,
        "input_ids": ids, "input_ids_sha256": id_digest(ids),
        "answers": example["answers"], "all_classes": example["all_classes"],
    })
validate_workload(SPEC, CASES)
write_json(OUTPUT / "workload.json", SPEC)
write_jsonl(OUTPUT / "cases.jsonl", CASES)
WORKLOAD_SHA = object_digest({"spec": SPEC, "cases": CASES})
print(f"Frozen {len(CASES)} prompts | {PRECISION} | workload SHA-256 {WORKLOAD_SHA}")
print(f'{sum(c["original_token_count"] > PROMPT_TOKEN_CAP for c in CASES)} prompts were middle-truncated.')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

Frozen 200 prompts | awq | workload SHA-256 94bd110fc970e318bb781284528fbfca667754f7fd1acf53a96b194a60de9ee1
153 prompts were middle-truncated.


## 3. Load one 3B model and check both cache paths

Use the authors’ SDPA attention and generation-preparation functions for the PyramidKV arm. Verify that every layer uses the authors’ selector and that greedy decoding reads the cache it produced. The five-prompt pilot includes the longest input and reports peak allocated GPU memory.

In [4]:
import sys
from contextlib import contextmanager

UPSTREAM_DIR = OUTPUT / "upstream"
UPSTREAM_DIR.mkdir(exist_ok=True)
if not (UPSTREAM_DIR / ".git").is_dir():
    subprocess.run(["git", "init", "-q", str(UPSTREAM_DIR)], check=True)
    subprocess.run(["git", "remote", "add", "origin", UPSTREAM_URL], cwd=UPSTREAM_DIR, check=True)
subprocess.run(["git", "sparse-checkout", "set", "pyramidkv"], cwd=UPSTREAM_DIR, check=True)
git = ["git", "-c", "http.version=HTTP/1.1"]
subprocess.run([*git, "fetch", "--depth=1", "--filter=blob:none", "origin", UPSTREAM_COMMIT],
               cwd=UPSTREAM_DIR, check=True)
subprocess.run([*git, "checkout", "--detach", "--quiet", "FETCH_HEAD"], cwd=UPSTREAM_DIR, check=True)
assert subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=UPSTREAM_DIR, text=True).strip() == UPSTREAM_COMMIT
sys.path.insert(0, str(UPSTREAM_DIR))

import pyramidkv.llama_model as official_llama
import pyramidkv.pyramidkv_utils as official_utils
from transformers import GenerationConfig
from transformers.models.llama.modeling_llama import LlamaForCausalLM, LlamaSdpaAttention

assert Path(official_llama.__file__).resolve().is_relative_to(UPSTREAM_DIR.resolve())
OFFICIAL_CONFIG = {
    "window_size": 8, "max_capacity_prompt": SPEC["budget"], "kernel_size": 7,
    "pooling": "maxpool", "merge": None, "kv_cache_granularity": "query_head",
}
OBSERVED_SELECTION = {}
BASE_SDPA_FORWARD = LlamaSdpaAttention.forward
BASE_PREPARE = LlamaForCausalLM.prepare_inputs_for_generation
BASE_UPDATE = official_utils.PyramidKVCluster.update_kv


def observe_official_update(cluster, key, query, value, attention_mask, groups):
    result = BASE_UPDATE(cluster, key, query, value, attention_mask, groups)
    assert cluster.layer_idx not in OBSERVED_SELECTION
    OBSERVED_SELECTION[int(cluster.layer_idx)] = (
        int(key.shape[-2]), int(result[0].shape[1]), int(result[0].shape[-2]),
    )
    return result


@contextmanager
def attention_mode(mode):
    if mode == "official_pyramidkv":
        # These are the authors' two SDPA-path assignments from replace_llama("pyramidkv").
        LlamaSdpaAttention.forward = official_llama.llama_sdpa_attn_forward_PyramidKV
        LlamaForCausalLM.prepare_inputs_for_generation = official_llama.prepare_inputs_for_generation_llama_new
        official_utils.PyramidKVCluster.update_kv = observe_official_update
    else:
        assert mode == "fullkv"
    try:
        yield
    finally:
        LlamaSdpaAttention.forward = BASE_SDPA_FORWARD
        LlamaForCausalLM.prepare_inputs_for_generation = BASE_PREPARE
        official_utils.PyramidKVCluster.update_kv = BASE_UPDATE
        OBSERVED_SELECTION.clear()


@torch.inference_mode()
def generate_case(model, case, mode, tokenizer, spec):
    OBSERVED_SELECTION.clear()
    input_ids = torch.tensor([case["input_ids"]], dtype=torch.long, device=model.device)
    generation = GenerationConfig(
        use_cache=True, return_dict_in_generate=True,
        do_sample=False, num_beams=1, max_new_tokens=spec["max_new_tokens"],
        eos_token_id=spec["stop_token_ids"], pad_token_id=spec["pad_token_id"],
    )
    result = model.generate(
        input_ids=input_ids, attention_mask=torch.ones_like(input_ids),
        past_key_values=DynamicCache(), generation_config=generation,
    )
    generated = result.sequences[0, input_ids.shape[-1]:].tolist()
    final = cache_shapes(result.past_key_values)
    if mode == "official_pyramidkv":
        assert set(OBSERVED_SELECTION) == set(range(NUM_LAYERS)), "The upstream selector did not run in every layer"
        assert all(before == input_ids.shape[-1] for before, _, _ in OBSERVED_SELECTION.values())
        kept = [OBSERVED_SELECTION[i][1:] for i in range(NUM_LAYERS)]
        assert all(heads == NUM_Q_HEADS for heads, _ in kept), "Expected the upstream query-head layout"
    else:
        assert not OBSERVED_SELECTION
        kept = [(NUM_KV_HEADS, input_ids.shape[-1])] * NUM_LAYERS
    return build_row(case, mode, generated, kept, final, tokenizer, spec)


MODEL = load_model(SPEC, "4.44.2")
assert all(isinstance(layer.self_attn, LlamaSdpaAttention) for layer in MODEL.model.layers)
for key, value in OFFICIAL_CONFIG.items():
    setattr(MODEL.config, key, value)
print(f"Loaded {PRECISION} Llama 3.2-3B on {GPU['name']} with the upstream SDPA path available.")

model.safetensors:   0%|          | 0.00/2.25G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/184 [00:00<?, ?B/s]

Loaded awq Llama 3.2-3B on Tesla T4 with the upstream SDPA path available.


In [5]:
def run_cases(cases, mode):
    results = []
    phase_dir = OUTPUT / "rows" / mode
    phase_dir.mkdir(parents=True, exist_ok=True)
    with attention_mode(mode):
        for offset, case in enumerate(cases, 1):
            row_path = phase_dir / f'{case["index"]:03d}.json'
            if row_path.is_file():
                row = read_json(row_path)
                validate_rows([row], [case], mode, SPEC)
            else:
                row = generate_case(MODEL, case, mode, TOKENIZER, SPEC)
                write_json(row_path, row)
            results.append(row)
            if offset == 1 or offset % 20 == 0 or offset == len(cases):
                print(f"{mode}: {offset}/{len(cases)}", flush=True)
    validate_rows(results, cases, mode, SPEC)
    return results


def run_pilot(variants):
    samples = pilot_cases(CASES)
    durations = {}
    for mode in variants:
        torch.cuda.reset_peak_memory_stats()
        start = time.perf_counter()
        with attention_mode(mode):
            results = [generate_case(MODEL, case, mode, TOKENIZER, SPEC) for case in samples]
        torch.cuda.synchronize()
        elapsed = time.perf_counter() - start
        validate_rows(results, samples, mode, SPEC)
        if mode != "fullkv":
            assert any(min(row["prefill_cache_tokens"]) < len(case["input_ids"])
                       for row, case in zip(results, samples)), "Pilot observed no eviction"
        durations[mode] = {
            "wall_seconds": round(elapsed, 3),
            "peak_allocated_mib": round(torch.cuda.max_memory_allocated() / 2**20, 1),
        }
    info = {"case_indices": [case["index"] for case in samples],
            "prompt_lengths": [len(case["input_ids"]) for case in samples],
            "by_mode": durations, "passed": True}
    write_json(OUTPUT / "pilot.json", info)
    print(json.dumps(info, indent=2))
    return info

MODES = ("fullkv", "official_pyramidkv")
PILOT = run_pilot(MODES)

{
  "case_indices": [
    58,
    38,
    184,
    96,
    116
  ],
  "prompt_lengths": [
    1884,
    4096,
    4096,
    4096,
    4096
  ],
  "by_mode": {
    "fullkv": {
      "wall_seconds": 76.662,
      "peak_allocated_mib": 5647.2
    },
    "official_pyramidkv": {
      "wall_seconds": 37.724,
      "peak_allocated_mib": 5540.6
    }
  },
  "passed": true
}


## 4. Run both paths on all 200 cases

The cell prints progress every 20 cases and saves a row after each completed case. If the cell is interrupted while the runtime is still connected, rerun this cell to resume.

In [6]:
assert PILOT["passed"]
for mode in MODES:
    write_jsonl(OUTPUT / f"{mode}.jsonl", run_cases(CASES, mode))
manifest = {
    "schema": SCHEMA, "arm": "official", "run_id": RUN_ID, "workload_sha256": WORKLOAD_SHA,
    "upstream_commit": UPSTREAM_COMMIT, "upstream_eviction": OFFICIAL_CONFIG,
    "gpu": GPU, "versions": package_versions(), "model_runtime": model_runtime(MODEL),
    "case_count": CASE_COUNT, "row_count": len(MODES) * CASE_COUNT,
    "pilot": PILOT,
}
write_json(OUTPUT / "run_manifest.json", manifest)
classifier = score_function()
for mode in MODES:
    rows = read_jsonl((OUTPUT / f"{mode}.jsonl").read_bytes())
    validate_rows(rows, CASES, mode, SPEC)
    mean = sum(score_prediction(classifier, row["prediction"], case)
               for row, case in zip(rows, CASES)) * 100 / CASE_COUNT
    print(f"{mode}: {mean:.2f}% on {CASE_COUNT} TREC cases")
OFFICIAL_ZIP = make_zip(OUTPUT, OFFICIAL_FILES)
print(f"Official result ZIP: {OFFICIAL_ZIP}")

fullkv: 1/200
fullkv: 20/200
fullkv: 40/200
fullkv: 60/200
fullkv: 80/200
fullkv: 100/200
fullkv: 120/200
fullkv: 140/200
fullkv: 160/200
fullkv: 180/200
fullkv: 200/200
official_pyramidkv: 1/200
official_pyramidkv: 20/200
official_pyramidkv: 40/200
official_pyramidkv: 60/200
official_pyramidkv: 80/200
official_pyramidkv: 100/200
official_pyramidkv: 120/200
official_pyramidkv: 140/200
official_pyramidkv: 160/200
official_pyramidkv: 180/200
official_pyramidkv: 200/200
fullkv: 67.50% on 200 TREC cases
official_pyramidkv: 65.50% on 200 TREC cases
Official result ZIP: /content/paper744_trec_official_20261122T183143Z-faaf2ef3.zip


## 5. Download the ZIP for Notebook B

If the browser blocks the download, locate the `paper744_trec_official_*.zip` file in the Colab Files sidebar.

In [7]:
from google.colab import files

files.download(str(OFFICIAL_ZIP))

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>